# 03a — Validate Article Bodies

**PURPOSE**: QA the article-body collection stage (03) before anything
downstream (place-candidate extraction, similarity search) is allowed to
consume it. Checks coverage against the cohort, PK integrity across the
three collected tables, null/parse-status/author-status distributions, and
block-type composition, and reports raw-HTML file presence/hash consistency.

**INPUT**:
- `data/20_processed/mbn/life/title_labels.parquet` (cohort definition, from 02b)
- `data/20_processed/mbn/life/article_body.parquet` (from 03)
- `data/20_processed/mbn/life/article_body_block.parquet` (from 03)
- `data/20_processed/mbn/life/article_authors.parquet` (from 03)
- `data/10_interim/mbn/life/fetch_attempts.parquet` (from 03)
- `data/00_raw/mbn/life/articles/*.html` (raw HTML files, from 03)

**OUTPUT**: `data/80_quality/article_body_validation_report.json`

**DEPENDENCIES**: pandas, pyarrow, `src.io.hashing`, `src.validation.checks`

**PARAMETERS**: none beyond the fixed input paths above (read entirely from
prior-stage outputs; no thresholds are hardcoded except the join-coverage
expectation, which is derived from `title_labels`, not a literal).

**ASSUMPTIONS**: 03 already enforced PK integrity on its own outputs at
write time; this notebook re-derives everything from disk (does not trust
in-memory state) so it is safe to run standalone after a kernel restart.

**SIDE EFFECTS**: writes the one report above; recomputes and verifies every
raw HTML file's SHA-256 against `article_body.bodySha256`'s companion path,
which touches disk but does not modify any existing artifact.

**FAIL CONDITIONS**: raises on PK violations found in any of the three 03
output tables (would indicate 03 itself is broken, not just incomplete
coverage). Coverage gaps, parse failures, and hash mismatches are recorded
as WARN/FAIL in `qualityStatus`, never silently passed.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (.git marker) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE


In [2]:
import json
from datetime import datetime, timezone

import pandas as pd

from src.io.hashing import sha256_file
from src.io.paths import data_dir, ensure_parent
from src.io.parquet_io import read_table
from src.validation.checks import assert_primary_key, duplicate_report, null_counts

LABELS_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "title_labels")
BODY_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_body")
BLOCK_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_body_block")
AUTHOR_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_authors")
ATTEMPTS_PATH = data_dir(REPO_ROOT, "10_interim", "mbn", "life", "fetch_attempts")
OUTPUT_QUALITY_PATH = data_dir(REPO_ROOT, "80_quality", "article_body_validation_report.json")

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

title_labels = read_table(LABELS_PATH)
article_body = read_table(BODY_PATH)
article_body_block = read_table(BLOCK_PATH)
article_authors = read_table(AUTHOR_PATH)
fetch_attempts = read_table(ATTEMPTS_PATH)

cohort_ids = set(title_labels.loc[~title_labels["isExcludedEditorialLabel"], "articleId"])
print("cohort size:", len(cohort_ids))
print("article_body:", article_body.shape, "| article_body_block:", article_body_block.shape)
print("article_authors:", article_authors.shape, "| fetch_attempts:", fetch_attempts.shape)

cohort size: 101
article_body: (101, 11) | article_body_block: (1619, 8)
article_authors: (101, 6) | fetch_attempts: (101, 8)


## Primary-key integrity (hard stop if 03's own outputs are internally broken)

In [3]:
assert_primary_key(article_body, ["articleId"], context="article_body")
assert_primary_key(article_body_block, ["articleId", "blockIndex"], context="article_body_block")
assert_primary_key(article_authors, ["articleId"], context="article_authors")
print("PK checks passed for article_body, article_body_block, article_authors")

PK checks passed for article_body, article_body_block, article_authors


## Coverage vs. cohort

In [4]:
body_ids = set(article_body["articleId"])
author_ids = set(article_authors["articleId"])
block_article_ids = set(article_body_block["articleId"]) if len(article_body_block) else set()

missing_from_body = cohort_ids - body_ids
extra_in_body = body_ids - cohort_ids
missing_from_author = cohort_ids - author_ids

coverage = {
    "cohort_size": len(cohort_ids),
    "article_body_rows": len(article_body),
    "article_authors_rows": len(article_authors),
    "missing_from_article_body": sorted(missing_from_body),
    "extra_in_article_body": sorted(extra_in_body),
    "missing_from_article_authors": sorted(missing_from_author),
    "articles_with_zero_blocks": sorted(cohort_ids & body_ids - block_article_ids),
}
for k, v in coverage.items():
    print(f"{k}: {v if not isinstance(v, list) else f'{len(v)} items: ' + str(v[:10])}")

cohort_size: 101
article_body_rows: 101
article_authors_rows: 101
missing_from_article_body: 0 items: []
extra_in_article_body: 0 items: []
missing_from_article_authors: 0 items: []
articles_with_zero_blocks: 0 items: []


## Parse-status / author-status / block-type distributions

In [5]:
parse_status_counts = article_body["parseStatus"].value_counts().to_dict()
author_status_counts = article_authors["authorParseStatus"].value_counts().to_dict()
author_method_counts = article_authors["authorParseMethod"].value_counts().to_dict()
block_type_counts = article_body_block["blockType"].value_counts().to_dict() if len(article_body_block) else {}

body_length_stats = article_body.loc[article_body["parseStatus"] != "fetch_failed", "bodyLength"].describe().to_dict()

print("parse_status_counts:", parse_status_counts)
print("author_status_counts:", author_status_counts)
print("author_method_counts:", author_method_counts)
print("block_type_counts:", block_type_counts)
print("body_length_stats:", body_length_stats)

parse_status_counts: {'ok': 101}
author_status_counts: {'found': 87, 'not_found': 12, 'partial': 2}
author_method_counts: {'b_tag_following_text': 86, 'not_found': 12, 'regex_email_only': 2, 'regex_tail_reporter_suffix': 1}
block_type_counts: {'paragraph': 1061, 'heading': 201, 'table': 168, 'image': 89, 'byline': 89, 'caption': 11}
body_length_stats: {'count': 101.0, 'mean': 1414.4851485148515, 'std': 816.5035776267167, 'min': 473.0, '25%': 1023.0, '50%': 1358.0, '75%': 1545.0, 'max': 5850.0}


## Raw HTML file presence + hash cross-check

In [6]:
hash_mismatches = []
missing_html_files = []

ok_rows = article_body[article_body["rawHtmlPath"].notna()]
for row in ok_rows.itertuples(index=False):
    html_path = REPO_ROOT / row.rawHtmlPath
    if not html_path.exists():
        missing_html_files.append(row.articleId)
        continue

blocks_by_article = article_body_block.groupby("articleId") if len(article_body_block) else None

for row in ok_rows.itertuples(index=False):
    if row.articleId in missing_html_files:
        continue
    if blocks_by_article is not None and row.articleId in blocks_by_article.groups:
        reconstructed = "\n".join(
            blocks_by_article.get_group(row.articleId)
            .loc[lambda d: d["blockType"] != "byline", "cleanText"]
        )
        if reconstructed != row.cleanBody:
            hash_mismatches.append(row.articleId)

print("missing_html_files:", len(missing_html_files), missing_html_files[:10])
print("cleanBody/blocks reconstruction mismatches:", len(hash_mismatches), hash_mismatches[:10])

missing_html_files: 0 []
cleanBody/blocks reconstruction mismatches: 0 []


## Fetch reliability (retries / timeouts / HTTP status distribution)

In [7]:
attempts_per_article = fetch_attempts.groupby("articleId").size()
retry_needed_count = int((attempts_per_article > 1).sum())
http_status_counts = fetch_attempts["httpStatus"].value_counts(dropna=False).to_dict()
error_type_counts = fetch_attempts["errorType"].value_counts(dropna=False).to_dict()

print("articles that needed a retry:", retry_needed_count, "/", len(attempts_per_article))
print("http_status_counts:", http_status_counts)
print("error_type_counts:", error_type_counts)

articles that needed a retry: 0 / 101
http_status_counts: {200: 101}
error_type_counts: {None: 101}


## Determine quality verdict and write report

In [8]:
blocking_issues = []
if missing_from_body:
    blocking_issues.append(f"{len(missing_from_body)} cohort articles missing from article_body")
if extra_in_body:
    blocking_issues.append(f"{len(extra_in_body)} article_body rows do not belong to the cohort")
if hash_mismatches:
    blocking_issues.append(f"{len(hash_mismatches)} articles have cleanBody/block reconstruction mismatches")

warnings = []
failed_count = parse_status_counts.get("fetch_failed", 0) + parse_status_counts.get("failed", 0) + parse_status_counts.get("partial", 0)
if failed_count:
    warnings.append(f"{failed_count} articles have non-'ok' parseStatus")
if missing_html_files:
    warnings.append(f"{len(missing_html_files)} raw HTML files missing on disk despite an 'ok' body row")
if author_status_counts.get("not_found", 0):
    warnings.append(f"{author_status_counts.get('not_found', 0)} articles have no parsed author")

quality_status = "FAIL" if blocking_issues else ("WARN" if warnings else "PASS")

report = {
    "notebook": "03aValidateArticleBodies.ipynb",
    "executionTimestamp": EXECUTION_TIMESTAMP,
    "coverage": coverage,
    "parseStatusCounts": parse_status_counts,
    "authorStatusCounts": author_status_counts,
    "authorMethodCounts": author_method_counts,
    "blockTypeCounts": block_type_counts,
    "bodyLengthStats": body_length_stats,
    "missingHtmlFiles": missing_html_files,
    "cleanBodyReconstructionMismatches": hash_mismatches,
    "retryNeededCount": retry_needed_count,
    "httpStatusCounts": {str(k): v for k, v in http_status_counts.items()},
    "errorTypeCounts": {str(k): v for k, v in error_type_counts.items()},
    "blockingIssues": blocking_issues,
    "warnings": warnings,
    "qualityStatus": quality_status,
}
ensure_parent(OUTPUT_QUALITY_PATH)
with open(OUTPUT_QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print("qualityStatus:", quality_status)
print("blockingIssues:", blocking_issues)
print("warnings:", warnings)
print("wrote:", OUTPUT_QUALITY_PATH.relative_to(REPO_ROOT))

qualityStatus: WARN
blockingIssues: []
warnings: ['12 articles have no parsed author']
wrote: data/80_quality/article_body_validation_report.json


## Closing summary

In [9]:
print("=== ROW COUNTS ===")
print({
    "article_body": len(article_body),
    "article_body_block": len(article_body_block),
    "article_authors": len(article_authors),
    "fetch_attempts": len(fetch_attempts),
})

print("=== NULL COUNTS ===")
print(null_counts(article_body, ["articleId", "cleanBody", "parseStatus"]))

print("=== DUPLICATES ===")
print(duplicate_report(article_body, [["articleId"]]))

print("=== QUALITY METRICS ===")
print({
    "parse_status_counts": parse_status_counts,
    "author_status_counts": author_status_counts,
    "missing_from_cohort": len(missing_from_body),
    "hash_mismatches": len(hash_mismatches),
})

print("=== OUTPUT PATH ===")
print(str(OUTPUT_QUALITY_PATH))

print("=== OUTPUT HASH ===")
from src.io.hashing import sha256_file as _sha
print({"report_sha256": _sha(OUTPUT_QUALITY_PATH)})

print("=== NEXT NOTEBOOK ===")
print("(none yet — next phase: place-candidate extraction from article_body_block, out of scope for this run)")

=== ROW COUNTS ===
{'article_body': 101, 'article_body_block': 1619, 'article_authors': 101, 'fetch_attempts': 101}
=== NULL COUNTS ===
{'articleId': 0, 'cleanBody': 0, 'parseStatus': 0}
=== DUPLICATES ===
{'articleId': 0}
=== QUALITY METRICS ===
{'parse_status_counts': {'ok': 101}, 'author_status_counts': {'found': 87, 'not_found': 12, 'partial': 2}, 'missing_from_cohort': 0, 'hash_mismatches': 0}
=== OUTPUT PATH ===
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/80_quality/article_body_validation_report.json
=== OUTPUT HASH ===
{'report_sha256': '7c10f0412a8e4f4f1d3bbc7af271770922bd36bfd0026c054ddf38230ad3056f'}
=== NEXT NOTEBOOK ===
(none yet — next phase: place-candidate extraction from article_body_block, out of scope for this run)
